In [1]:
# bowaka_v2_lab notebook bootstrap cell — DO NOT EDIT BY HAND.
# Adds the lab's src/ (and its bowaka_common dependency) to sys.path and pins
# the working directory to the repo root, so `import bowaka_v2_lab` and
# repo-root-relative CONFIG_PATH parameters resolve identically under jupyter,
# papermill, and the QuantsLab scheduler.
import os
import sys
from pathlib import Path

_lab_root = None
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "bowaka_v2_lab" / "__init__.py").is_file():
        _lab_root = _candidate
        break
if _lab_root is None:
    raise RuntimeError(
        f"bowaka_v2_lab bootstrap: src/bowaka_v2_lab/ not found at or above {Path.cwd()}"
    )

# Pin CWD to the repo root (the directory holding research_notebooks/ and the
# Makefile) so repo-root-relative CONFIG_PATH values resolve regardless of how
# the notebook was launched (jupyter CWD = notebook dir, scheduler = repo root).
_repo_root = _lab_root
for _candidate in [_lab_root, *_lab_root.parents]:
    if (_candidate / "research_notebooks").is_dir() and (_candidate / "Makefile").is_file():
        _repo_root = _candidate
        break
os.chdir(_repo_root)

# Make the lab and its bowaka_common dependency importable from the working
# tree, even when the packages are not pip-installed. v1 bowaka_lab is
# deliberately excluded — v2 must not import v1.
for _src in (_lab_root / "src",
             _repo_root / "research_notebooks" / "bowaka_common" / "src"):
    if _src.is_dir() and str(_src) not in sys.path:
        sys.path.insert(0, str(_src))

import bowaka_v2_lab  # noqa: F401
print(f"bowaka_v2_lab {bowaka_v2_lab.__version__} (cwd={_repo_root})")


bowaka_v2_lab 0.1.0 (cwd=/quants-lab)


In [2]:
# Papermill parameters.
CONFIG_PATH = 'research_notebooks/bowaka_v2_lab/configs/bowaka_v2_walkforward_optuna.yml'
N_TRIALS = None          # None -> optuna.n_trials from the config; or set an integer
N_STARTUP_TRIALS = None  # None -> optuna.n_startup_trials; random trials before TPE


# 10 — Walk-Forward Optuna

Runs a **real** walk-forward parameter optimization against the shared
market-data lake. Each Optuna trial samples a parameter set, applies it
to the config, and runs a real backtest over every walk-forward
validation window; the trial objective is the median fold score. The
final-holdout window is never read during tuning.

**Parameters:** `N_TRIALS` is the total trial count (`None` -> the
config's `optuna.n_trials`). `N_STARTUP_TRIALS` is how many of those are
random-sampling trials before TPE-guided search begins (`None` -> the
config's `optuna.n_startup_trials`).

**Compute:** a run is `N_TRIALS` x `n_folds` real backtests — the config
default (2500 trials) is a multi-day job. Set a small `N_TRIALS` and a
focused `universe.symbols` for a quick run.

In [ ]:
import json
from bowaka_v2_lab.optuna.walkforward_runner import run_walkforward_study
result = run_walkforward_study(CONFIG_PATH, n_trials=N_TRIALS,
  n_startup_trials=N_STARTUP_TRIALS)
print(json.dumps(result, indent=2, default=str))


/quants-lab/research_notebooks/bowaka_v2_lab/src/bowaka_v2_lab/optuna/dispatcher.py:46: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  sampler=optuna.samplers.TPESampler(
[I 2026-05-21 07:46:17,972] Using an existing study with name 'bowaka_v2_iex_walkforward_conservative_7cfc7498_20260521' instead of creating a new one.
2026-05-21 07:46:18,003 INFO walk-forward study bowaka_v2_iex_walkforward_conservative_7cfc7498_20260521: 5000 trials (500 random startup) x 21 folds, 100 symbols, feed=iex
[I 2026-05-21 08:13:34,516] Trial 6 finished with value: -1.0 and parameters: {'signals.rvol_so_far_min': 2.0480987000623267, 'signals.projected_full_day_rvol_min': 1.6347358886178625, 'signals.range_expansion_so_far_min': 1.6953162987358994, 'signals.close_location_so_far_min': 0.5985559766894682, 'signals.ema_distance_min': 0.06235018918205848, 'signals.ema_slope_min': 0.04220713847170446, 'signals.gap_pct_max': 0.14573260427126

KeyboardInterrupt: 